In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")


In [2]:
from dataclasses import dataclass

@dataclass
class ColourContext:
    favourite_colour: str = "blue"
    least_favourite_colour: str = "yellow"

In [3]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent  # ← back to this

model = init_chat_model(
    "llama-3.1-8b-instant",  # ← official replacement for the decommissioned model
    model_provider="groq",
    temperature=0,  # ← MUST be 0 for reliable tool calling
)

agent = create_agent(
    model=model, 
    context_schema=ColourContext,
    )


In [4]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

In [5]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='926d607a-e02a-4040-820c-bb286a2b568a'),
              AIMessage(content="I don't have any information about your personal preferences, including your favorite color. I'm a large language model, I don't have the ability to retain information about individual users or their preferences. Each time you interact with me, it's a new conversation and I don't have any prior knowledge about you. If you'd like to share your favorite color, I'd be happy to chat with you about it!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 84, 'prompt_tokens': 41, 'total_tokens': 125, 'completion_time': 0.099990238, 'completion_tokens_details': None, 'prompt_time': 0.003033089, 'prompt_tokens_details': None, 'queue_time': 0.006574543, 'total_time': 0.103023327}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_020e283281', 'service_tier': 'on_demand'

## Accessing Context

In [6]:
from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the favourite colour of the user"""
    return runtime.context.favourite_colour

@tool
def get_least_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour

In [7]:
agent = create_agent(
    model=model,
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext
)

In [8]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

pprint(response)

F:\DEV\langchain-ai\lca-lc-foundations\.venv\Lib\site-packages\pydantic\functional_validators.py:839: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  function=lambda v, h: h(v), schema=original_schema
F:\DEV\langchain-ai\lca-lc-foundations\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(
F:\DEV\langchain-ai\lca-lc-foundations\.venv\Lib\site-packages\pydantic\functional_validators.py:839: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none`

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='bede78b1-dd6c-4a38-ad57-a0203d3b9831'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'nge8xq17f', 'function': {'arguments': '{}', 'name': 'get_favourite_colour'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 267, 'total_tokens': 278, 'completion_time': 0.026367841, 'completion_tokens_details': None, 'prompt_time': 0.018774449, 'prompt_tokens_details': None, 'queue_time': 0.006019932, 'total_time': 0.04514229}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e2c608b1d6', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f3d96-dbfb-7813-843e-36edc367828c-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}, 'id': 'nge8xq17f', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_toke

In [9]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(favourite_colour="green")
)

pprint(response)

F:\DEV\langchain-ai\lca-lc-foundations\.venv\Lib\site-packages\pydantic\functional_validators.py:839: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  function=lambda v, h: h(v), schema=original_schema
F:\DEV\langchain-ai\lca-lc-foundations\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(
F:\DEV\langchain-ai\lca-lc-foundations\.venv\Lib\site-packages\pydantic\functional_validators.py:839: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none`

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='978d7761-4645-424b-9d2b-6724266c4953'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'r4y3yhv6g', 'function': {'arguments': '{}', 'name': 'get_favourite_colour'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 267, 'total_tokens': 278, 'completion_time': 0.024956166, 'completion_tokens_details': None, 'prompt_time': 0.067931503, 'prompt_tokens_details': None, 'queue_time': 0.03694263, 'total_time': 0.092887669}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e2c608b1d6', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f3d9a-2f3f-72e3-b852-68caa53e2fd1-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}, 'id': 'r4y3yhv6g', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_toke